# Ken French Library Demo / Ken French 因子库示例

这个 notebook 下载日频 Fama-French 五因子和动量因子，把它们拼成一张表，再看因子的长期表现。  
This notebook downloads the daily Fama-French five-factor dataset plus the daily momentum factor, joins them into one table, and visualizes the factor history.

它更适合做解释和稳健性检验，而不是 Track A 的主特征源。  
It is better suited to interpretation and robustness checks than to being the primary feature source for Track A.


In [ ]:
import importlib.util

# 只检查 pandas_datareader 依赖，不在 notebook 内自动安装，避免 Windows/Jupyter 子进程编码问题。
# Only check for pandas_datareader instead of auto-installing it inside the notebook, which avoids Windows/Jupyter subprocess encoding issues.
if importlib.util.find_spec("pandas_datareader") is None:
    raise ImportError("Missing package: pandas_datareader. 请先安装：pip install pandas_datareader")

import matplotlib.pyplot as plt
import pandas as pd
import pandas_datareader.data as web
from IPython.display import display

plt.style.use("ggplot")


In [ ]:
# 把五因子和动量因子合在一起，统一成日频百分比表。
# Join the five-factor table and the momentum factor into one daily percentage table.
ff5 = web.DataReader("F-F_Research_Data_5_Factors_2x3_daily", "famafrench")
mom = web.DataReader("F-F_Momentum_Factor_daily", "famafrench")

ff5_table = ff5[0].copy()
ff5_table.columns = [col.strip() for col in ff5_table.columns]

mom_table = mom[0].copy()
mom_table.columns = [col.strip() for col in mom_table.columns]

factors = ff5_table.join(mom_table[["Mom"]], how="inner")
factors.index = pd.to_datetime(factors.index.astype(str))

display(factors.tail())
display(factors.describe().T[["mean", "std", "min", "max"]])


In [ ]:
# 一张图看累计表现，另一张图看 63 日滚动均值，方便感受因子状态。
# Use one chart for cumulative performance and another for the 63-day rolling mean to get a feel for factor regimes.
factor_cols = ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom"]
cumulative = (1 + factors[factor_cols] / 100).cumprod()
rolling_mean = factors[factor_cols].rolling(63).mean()

fig, axes = plt.subplots(2, 1, figsize=(12, 10))

cumulative.plot(ax=axes[0], linewidth=1.8)
axes[0].set_title("Cumulative factor performance / 因子累计表现")
axes[0].set_ylabel("Growth of 1 / 单位净值")

rolling_mean.plot(ax=axes[1], linewidth=1.4)
axes[1].set_title("63-day rolling factor mean / 63 日滚动因子均值")
axes[1].set_ylabel("Daily factor return (%) / 日度因子收益(%)")

plt.tight_layout()
plt.show()


## Why it matters / 为什么它有用

- 因子数据可以帮助解释 HMM regime 更像“价值修复”还是“风险偏好扩张”。  
  Factor data helps interpret whether an HMM regime looks more like a value recovery or a broad risk-on expansion.
- 它也适合做 robustness check，看不同 regime 下因子暴露有没有系统差异。  
  It is also useful for robustness checks, such as testing whether factor exposures differ systematically across regimes.
